<a href="https://colab.research.google.com/github/BlackJackBander/HeadHunter_bot/blob/main/headhunter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import asyncio
import aiohttp
import csv
import re

# Настройки поиска (можно вынести в переменные окружения)
KEYWORDS = 'NAME:(DevOps OR "Системный администратор") AND Python'
EXPERIENCE = 'between1And3'  # Доступно: noExperience, between1And3, between3And6, moreThan6
SCHEDULE = 'remote'          # Доступно: fullDay, shift, flexible, remote, flyInFlyOut
AREA = 113                   # 113 - вся Россия
MAX_PAGES = 3                # Сколько страниц результатов обходить
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36'



def clean_html(raw_html):
    if not raw_html: return ""
    clean_text = re.sub(r'<[^>]+>', ' ', raw_html)
    return ' '.join(clean_text.split())

async def fetch_vacancy_detail(session, vacancy_id, seen_ids):
    """Получает детальную информацию по ID вакансии."""
    if vacancy_id in seen_ids:
        return None

    url = f'https://api.hh.ru/vacancies/{vacancy_id}'
    async with session.get(url) as response:
        if response.status == 200:
            seen_ids.add(vacancy_id)
            data = await response.json()

            # Извлекаем данные
            name = data.get('name', '')
            area = data.get('area', {}).get('name', '')
            skills = ", ".join([s['name'] for s in data.get('key_skills', [])])
            description = clean_html(data.get('description', ''))
            alt_url = data.get('alternate_url', '')

            # Обработка зарплаты
            sal = data.get('salary')
            salary_str = f"{sal.get('from')}-{sal.get('to')} {sal.get('currency')}" if sal else "Не указана"

            return [name, salary_str, skills, area, alt_url, description]
        return None

async def main():
    headers = {'User-Agent': USER_AGENT}
    seen_ids = set() # Множество для проверки дубликатов
    all_results = []

    async with aiohttp.ClientSession(headers=headers) as session:
        tasks = []

        print(f"Сбор списка вакансий по запросу: {KEYWORDS}...")

        for page in range(MAX_PAGES):
            params = {
                'text': KEYWORDS,
                'experience': EXPERIENCE,
                'schedule': SCHEDULE,
                'area': AREA,
                'page': page,
                'per_page': 20
            }

            async with session.get('https://api.hh.ru/vacancies', params=params) as response:
                if response.status != 200:
                    print(f"Ошибка поиска на странице {page}: {response.status}")
                    continue

                search_data = await response.json()
                items = search_data.get('items', [])

                for item in items:
                    # Создаем задачу на детальный парсинг
                    tasks.append(fetch_vacancy_detail(session, item['id'], seen_ids))

        # Запускаем все задачи одновременно
        print(f"Загрузка деталей для {len(tasks)} уникальных вакансий...")
        results = await asyncio.gather(*tasks)

        # Фильтруем пустые результаты (где были ошибки или дубли)
        all_results = [r for r in results if r is not None]

    # Сохранение
    if all_results:
        with open('hh_async_results.csv', 'w', encoding='utf-8-sig', newline='') as f:
            writer = csv.writer(f, delimiter=';')
            writer.writerow(['Название', 'Зарплата', 'Навыки', 'Город', 'Ссылка', 'Описание'])
            writer.writerows(all_results)
        print(f"Готово! Сохранено {len(all_results)} вакансий.")
    else:
        print("Новых вакансий не найдено.")

# if __name__ == '__main__':
#     asyncio.run(main())

# Fix for RuntimeError: asyncio.run() cannot be called from a running event loop in Colab/Jupyter
await main()

Сбор списка вакансий по запросу: NAME:(DevOps OR "Системный администратор") AND Python...
Загрузка деталей для 41 уникальных вакансий...
Готово! Сохранено 41 вакансий.
